In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

EXCLURE_DEJA_DEMENTS = True   # meme choix que Membre A (notebooks 10, 14, 15)

prep = sorted(C.COHORTE.glob("cible_ancree_359_*"))[-1]
jeu = pd.read_csv(prep / "jeu_modelisation_359_ancre.csv")
print("Donnees :", prep.name, "|", jeu.shape)

donnees = jeu.dropna(subset=["progression_ancree"]).copy()
if EXCLURE_DEJA_DEMENTS:
    n0 = len(donnees)
    donnees = donnees.loc[donnees["deja_dement"] == 0].copy()
    print("Patients deja dements a T0 exclus :", n0 - len(donnees))

TAB_CANDIDATES = [
    "sexe", "education", "age_reference", "APOE_e4",
    "CDR_global", "CDR_SB", "MMSE_debut", "ADAS13", "FAQ_debut",
    "ABETA42", "TAU", "PTAU", "dx_T0",
]
FEATURES = [c for c in TAB_CANDIDATES if donnees[c].notna().sum() > 0]
vides = [c for c in TAB_CANDIDATES if c not in FEATURES]
if vides:
    print("Variables ignorees (vides) :", vides)

X = donnees[FEATURES]
y = donnees["progression_ancree"].astype(int)
print("Participants :", len(y), "| progressent :", int(y.sum()),
      f"({y.mean() * 100:.1f}%)")
print("Variables utilisees :", FEATURES)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

imp = lambda: SimpleImputer(strategy="median")
avec = lambda clf: Pipeline([("imp", imp()), ("sc", StandardScaler()), ("clf", clf)])
sans = lambda clf: Pipeline([("imp", imp()), ("clf", clf)])

MODELES = {
    "Hasard (reference)": sans(DummyClassifier(strategy="most_frequent")),
    "Regression logistique": avec(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Arbre de decision": sans(DecisionTreeClassifier(class_weight="balanced", random_state=0)),
    "Random forest": sans(RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0)),
    "Gradient boosting": sans(HistGradientBoostingClassifier(random_state=0)),
    "SVM (noyau RBF)": avec(SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=0)),
    "k plus proches voisins": avec(KNeighborsClassifier(n_neighbors=15)),
}

SCORES = ["roc_auc", "balanced_accuracy", "f1", "accuracy"]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

brut = {}
for nom, m in MODELES.items():
    res = cross_validate(m, X, y, cv=CV, scoring=SCORES)
    brut[nom] = {s: (res["test_" + s].mean(), res["test_" + s].std()) for s in SCORES}

noms_fr = {"roc_auc": "ROC-AUC", "balanced_accuracy": "Exactitude equilibree",
           "f1": "F1", "accuracy": "Exactitude"}
tableau = pd.DataFrame({
    noms_fr[s]: {n: f"{brut[n][s][0]:.3f} ± {brut[n][s][1]:.3f}" for n in MODELES}
    for s in SCORES})
ordre = sorted(MODELES, key=lambda n: brut[n]["roc_auc"][0], reverse=True)
tableau = tableau.loc[ordre]
tableau.index.name = "Modele"

print(f"=== Comparaison des modeles — cible ANCREE, {len(y)} participants ===")
display(tableau)

In [ ]:
# --- Graphique + choix final argumente ---
moyennes = [brut[n]["roc_auc"][0] for n in ordre]
ecarts = [brut[n]["roc_auc"][1] for n in ordre]

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
pos = np.arange(len(ordre))[::-1]
ax.barh(pos, moyennes, xerr=ecarts, color="#4C72B0", capsize=4)
ax.set_yticks(pos); ax.set_yticklabels(ordre)
ax.set_xlabel("ROC-AUC (validation croisee 5 plis)")
ax.set_xlim(0.40, 1.0)
ax.axvline(0.5, color="grey", ls="--", lw=1, label="hasard")
for p, m, e in zip(pos, moyennes, ecarts):
    ax.text(m + e + 0.012, p, f"{m:.3f}", va="center", fontsize=9)
ax.set_title("Comparaison des modeles — cible ancree")
ax.legend(loc="lower right")

SORTIE = C.sortie("comparaison_modeles_ancree")
fig.savefig(SORTIE / "comparaison_AUC_ancree.png", dpi=150)
plt.show()

reels = {n: brut[n]["roc_auc"] for n in MODELES if n != "Hasard (reference)"}
meilleur = max(reels, key=lambda n: reels[n][0])
auc_m = reels[meilleur][0]
auc_log = reels["Regression logistique"][0]
if meilleur != "Regression logistique" and (auc_m - auc_log) <= 0.010:
    choix = "Regression logistique"
    raison = (f"a seulement {auc_m - auc_log:.3f} d'AUC du meilleur ({meilleur}), "
              "mais plus simple et interpretable")
else:
    choix, raison = meilleur, "meilleure ROC-AUC"

print("Classement par ROC-AUC :")
for n in ordre:
    if n != "Hasard (reference)":
        print(f"  {n:28s} AUC = {brut[n]['roc_auc'][0]:.3f} ± {brut[n]['roc_auc'][1]:.3f}")
print(f"\n>>> MODELE RETENU : {choix}")
print(f"    Justification : {raison}.")

tableau.to_csv(SORTIE / "comparaison_modeles_ancree.csv", encoding="utf-8-sig")
pd.DataFrame([{"modele": n, "roc_auc_moyenne": brut[n]["roc_auc"][0],
               "roc_auc_ecart_type": brut[n]["roc_auc"][1]} for n in ordre]
             ).to_csv(SORTIE / "classement_AUC_ancree.csv", index=False, encoding="utf-8-sig")
print("\nResultats enregistres dans :", SORTIE)

In [ ]:
# --- Avant / apres correction de la cible (pour la presentation) ---
ancien = sorted(C.COHORTE.glob("comparaison_modeles_[0-9]*"))
if ancien:
    avant = pd.read_csv(ancien[-1] / "classement_AUC.csv")
    apres = pd.read_csv(SORTIE / "classement_AUC_ancree.csv")
    comp = (avant[["modele", "roc_auc_moyenne"]].rename(columns={"roc_auc_moyenne": "AUC cible initiale"})
            .merge(apres[["modele", "roc_auc_moyenne"]].rename(
                columns={"roc_auc_moyenne": "AUC cible ancree"}), on="modele"))
    comp["ecart"] = (comp["AUC cible ancree"] - comp["AUC cible initiale"]).round(3)
    display(comp.round(3))
    print("La baisse est attendue : la cible initiale incluait des patients deja aggraves.")
else:
    print("Resultats du notebook 11 introuvables : lance-le d'abord si tu veux la comparaison.")